In [9]:
import pandas as pd
import json
import os
os.chdir('/Users/sophiaye/Desktop/personal_projects/buses')

In [ ]:
index_df  = pd.read_csv('merge_datasets/bus_need_index_final.csv')
route_nta = pd.read_csv('merge_datasets/route_nta_mapping.csv').dropna(subset=['NTACode'])

top_ntas = index_df.sort_values('Score', ascending=False).head(20)['NTACode'].tolist()

route_coverage = (
    route_nta[route_nta['NTACode'].isin(top_ntas)]
    .groupby('route_id')['NTACode'].nunique()
    .reset_index()
    .rename(columns={'NTACode': 'high_need_count'})
)

priority_routes  = set(route_coverage[route_coverage['high_need_count'] >= 3]['route_id'])
secondary_routes = set(route_coverage[route_coverage['high_need_count'].between(1, 2)]['route_id'])

print(f"Priority routes ({len(priority_routes)}):  {sorted(priority_routes)}")
print(f"Secondary routes ({len(secondary_routes)}): {sorted(secondary_routes)[:10]}...")

target_routes = priority_routes | secondary_routes



Priority routes (23):  ['B11', 'B16', 'B35', 'B49', 'B6', 'B8', 'B9', 'BX19', 'BX9', 'M1', 'M100', 'M101', 'M102', 'M11', 'M125', 'M2', 'M3', 'M4', 'M5', 'M7', 'Q44+', 'Q58', 'Q98']
Secondary routes (56): ['B1', 'B12', 'B36', 'B4', 'B41', 'B44', 'B44+', 'B68', 'B70', 'B82']...

Routes with shape mapping: 79


In [ ]:
gtfs_folders = [
    'nta_bus_mapping/data/gtfs_b',
    'nta_bus_mapping/data/gtfs_bx',
    'nta_bus_mapping/data/gtfs_m',
    'nta_bus_mapping/data/gtfs_q',
    'nta_bus_mapping/data/gtfs_si',
]

all_trips  = []
all_shapes = []

for folder in gtfs_folders:
    trips_path  = os.path.join(folder, 'trips.txt')
    shapes_path = os.path.join(folder, 'shapes.txt')
    if os.path.exists(trips_path):
        t = pd.read_csv(trips_path, usecols=['route_id', 'shape_id']).drop_duplicates()
        all_trips.append(t)
    if os.path.exists(shapes_path):
        s = pd.read_csv(shapes_path)
        all_shapes.append(s)

trips_df  = pd.concat(all_trips,  ignore_index=True).drop_duplicates()
shapes_df = pd.concat(all_shapes, ignore_index=True)

trips_filtered = trips_df[trips_df['route_id'].isin(target_routes)]

route_to_shape = (
    trips_filtered
    .sort_values('shape_id')
    .drop_duplicates(subset='route_id', keep='first')
    .set_index('route_id')['shape_id']
    .to_dict()
)

print(f"\nRoutes with shape mapping: {len(route_to_shape)}")

In [ ]:
features = []

for route_id, shape_id in route_to_shape.items():
    pts = (
        shapes_df[shapes_df['shape_id'] == shape_id]
        .sort_values('shape_pt_sequence')[['shape_pt_lon', 'shape_pt_lat']]
        .values.tolist()
    )
    if len(pts) < 2:
        continue

    tier = 'priority' if route_id in priority_routes else 'secondary'
    high_need_count = int(route_coverage[route_coverage['route_id'] == route_id]['high_need_count'].values[0]) if route_id in route_coverage['route_id'].values else 0

    features.append({
        'type': 'Feature',
        'properties': {
            'route_id':        route_id,
            'tier':            tier,
            'high_need_count': high_need_count
        },
        'geometry': {
            'type':        'LineString',
            'coordinates': pts
        }
    })

geojson = {'type': 'FeatureCollection', 'features': features}

out_path = 'nta_map/bus_routes.geojson'
with open(out_path, 'w') as f:
    json.dump(geojson, f)

print(f"\nExported {len(features)} routes to {out_path}")
print(f"  Priority:  {sum(1 for ft in features if ft['properties']['tier'] == 'priority')}")
print(f"  Secondary: {sum(1 for ft in features if ft['properties']['tier'] == 'secondary')}")


Exported 79 routes to nta_map/bus_routes.geojson
  Priority:  23
  Secondary: 56
